<a href="https://colab.research.google.com/github/popcorn0125/Solar_energy_prediction/blob/main/AI_DX_EDU_%ED%83%9C%EC%96%91%EA%B4%91_%EC%A0%84%EB%A0%A5%EB%9F%89_test_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%88%98%EC%A7%91.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [데이터 전처리] 대용량 파일 분할 및 정제 (Solar Only)

2017년부터 2023년 2월까지의 데이터가 통합된 파일에서 **풍력 데이터는 제거**하고, **태양광 데이터만 남겨 연도별로 분리 저장**합니다.

### 📋 수행 기능
1.  **데이터 로드**: 업로드된 `230403_지역별 시간별 태양광 발전량.csv` 파일을 불러옵니다.
2.  **컬럼 정리**:
    * 공백이 포함된 지저분한 컬럼명을 깔끔하게 변경합니다. (`' 태양광 발전량(MWh) '` -> `발전량`)
    * 불필요한 `' 풍력 발전량(MWh) '` 컬럼은 **삭제**합니다.
3.  **데이터 정제**:
    * 빈 값(결측치)이나 공백으로 비어있는 발전량 데이터를 `0`으로 채웁니다.
4.  **연도별 분할 저장**:
    * `solar_trade_2017.csv` ~ `solar_trade_2023.csv` 파일로 각각 나누어 저장합니다.
    * 특히 `solar_trade_2023.csv`에는 우리가 필요했던 **2023년 1월, 2월 데이터**가 담기게 됩니다.

In [ ]:
import pandas as pd
import os

# -----------------------------------------------------------
# [입력] 업로드한 통합 파일명을 정확히 입력하세요.
# -----------------------------------------------------------
BIG_FILE_NAME = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/230403_지역별 시간별 태양광 발전량.csv'

def split_solar_data_by_year():
    # 1. 파일 확인
    if not os.path.exists(BIG_FILE_NAME):
        print(f"[오류] '{BIG_FILE_NAME}' 파일을 찾을 수 없습니다.")
        return

    print(f"파일 로드 중... ({BIG_FILE_NAME})")

    # 2. 데이터 읽기 (인코딩 자동 처리)
    try:
        df = pd.read_csv(BIG_FILE_NAME, encoding='cp949')
    except UnicodeDecodeError:
        df = pd.read_csv(BIG_FILE_NAME, encoding='utf-8')

    print(f"원본 데이터 개수: {len(df)}개")
    print(f"원본 컬럼명: {df.columns.tolist()}")

    # 3. 컬럼명 공백 제거 및 정리
    # 컬럼명 앞뒤 공백 제거 (' 태양광... ' -> '태양광...')
    df.columns = df.columns.str.strip()

    # 필요한 컬럼만 선택 ('풍력' 제거)
    # 파일에 있는 정확한 컬럼명: '거래일자', '거래시간', '지역', '태양광 발전량(MWh)'
    target_cols = ['거래일자', '거래시간', '지역', '태양광 발전량(MWh)']

    # 만약 컬럼명이 조금 다를 수 있으니 확인 후 선택
    available_cols = [c for c in target_cols if c in df.columns]
    df = df[available_cols].copy()

    # 컬럼 이름 변경 (직관적으로 '발전량'으로 통일)
    df.rename(columns={'태양광 발전량(MWh)': '발전량'}, inplace=True)

    # 4. 데이터 정제 (결측치 처리)
    # 공백이나 NaN을 0으로 채움
    df['발전량'] = pd.to_numeric(df['발전량'], errors='coerce').fillna(0)

    # 5. 연도 추출
    df['거래일자'] = pd.to_datetime(df['거래일자'])
    df['Year'] = df['거래일자'].dt.year

    # 6. 연도별 분할 및 저장
    unique_years = df['Year'].unique()
    print(f"\n발견된 연도: {sorted(unique_years)}")

    created_files = []

    for year in sorted(unique_years):
        # 해당 연도 데이터 추출
        year_df = df[df['Year'] == year].copy()
        year_df.drop(columns=['Year'], inplace=True) # 임시 컬럼 삭제

        # 파일명 생성
        save_name = f"/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_{year}.csv"
        year_df.to_csv(save_name, index=False, encoding='utf-8-sig')

        print(f"  -> [저장 완료] {save_name} ({len(year_df)}개 행)")
        created_files.append(save_name)

    print("\n[완료] 총 {}개의 파일이 생성되었습니다.".format(len(created_files)))
    if 'solar_trade_2023.csv' in created_files:
        print(" 성공: 'solar_trade_2023.csv' (1~2월 데이터 포함) 확보 완료!")

# 함수 실행
split_solar_data_by_year()

파일 로드 중... (/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/230403_지역별 시간별 태양광 발전량.csv)
원본 데이터 개수: 918000개
원본 컬럼명: ['거래일자', '거래시간', '지역', ' 태양광 발전량(MWh) ', ' 풍력 발전량(MWh) ']

발견된 연도: [np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]
  -> [저장 완료] /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_2017.csv (148920개 행)
  -> [저장 완료] /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_2018.csv (148920개 행)
  -> [저장 완료] /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_2019.csv (148920개 행)
  -> [저장 완료] /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_2020.csv (149328개 행)
  -> [저장 완료] /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_2021.csv (148920개 행)
  -> [저장 완료] /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/solar_trade_2022.csv (148920개 행)
  -> [저장 완료] /c

# 8-3. [최종 병합] 2023년 태양광 데이터 완성 (Jan~Dec)

여러 개로 쪼개진 2023년 파일들을 하나로 합쳐 **검증용 데이터셋(Test Set)**을 완성합니다.
1월~2월 데이터는 통합 파일에서 추출하고, 나머지 3월~12월 데이터는 각 월별 파일에서 가져와 연결합니다.

### 📋 수행 기능
1.  **파일별 맞춤 처리**:
    * `230403...csv`: 2023년 데이터(1~2월)만 뽑아냅니다.
    * 나머지 파일들: 3월~12월 데이터를 그대로 가져옵니다.
2.  **데이터 정제 (스마트 필터링)**:
    * **지역**: '대구' 데이터만 남깁니다. (컬럼명이 `지역`이든 `지역명`이든 자동 인식)
    * **태양광 추출**: `연료원` 컬럼이 있으면 '태양광'을 필터링하고, 없으면 `태양광 발전량` 컬럼을 찾아냅니다. (풍력 데이터 제거)
3.  **최종 저장**:
    * `daegu_solar_2023_test.csv` 파일로 저장하여, 바로 백테스팅에 사용할 수 있도록 합니다.

In [ ]:
import pandas as pd
import os

# -----------------------------------------------------------
# [입력] 업로드된 파일들의 이름을 리스트에 적어줍니다.
# -----------------------------------------------------------
files_info = [
    # 1. 1~2월 데이터가 들어있는 통합 파일
    {'name': '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/한국전력거래소_지역별 시간별 태양광 발전량_202301_202302.csv', 'type': 'mixed'},

    # 2. 나머지 월별 파일들 (3월 ~ 12월)
    {'name': '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/한국전력거래소_지역별 시간별 태양광 발전량_20230301_20230531.csv', 'type': 'part'},
    {'name': '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/지역별 시간별 태양광 및 풍력 발전량_20230601_20230831.csv', 'type': 'part'},
    {'name': '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/지역별 시간대별 태양광 및 풍력 발전량(2309_2311).csv', 'type': 'part'},
    {'name': '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/한국전력거래소_지역별 시간별 태양광 및 풍력 발전량_20231231.csv', 'type': 'part'}
]

def create_full_2023_dataset():
    all_dfs = []
    print("데이터 병합 시작...\n")

    for file_info in files_info:
        fname = file_info['name']
        ftype = file_info['type']

        if not os.path.exists(fname):
            print(f" 파일 없음(건너뜀): {fname}")
            continue

        try:
            # 파일 읽기
            try:
                df = pd.read_csv(fname, encoding='cp949')
            except UnicodeDecodeError:
                df = pd.read_csv(fname, encoding='utf-8')

            # 컬럼명 공백 제거
            df.columns = df.columns.str.strip()

            # 1. 지역 필터링 ('지역' 또는 '지역명')
            region_col = '지역' if '지역' in df.columns else '지역명'
            if region_col in df.columns:
                df = df[df[region_col].str.contains('대구', na=False)].copy()

            # 2. 태양광 데이터만 남기기 (풍력 제거)
            # Case A: '연료원' 컬럼이 있는 경우 (예: 12월 파일)
            if '연료원' in df.columns:
                df = df[df['연료원'] == '태양광'].copy()
                # 발전량 컬럼 찾기
                solar_col = [c for c in df.columns if '발전량' in c or '전력거래량' in c][0]

            # Case B: '태양광 발전량' 등 전용 컬럼이 있는 경우
            else:
                solar_cols = [c for c in df.columns if ('태양광' in c or '발전량' in c)]
                if solar_cols:
                    solar_col = solar_cols[0]
                else:
                    print(f"   태양광 컬럼 못 찾음: {fname}")
                    continue

            # 3. 컬럼 이름 통일 ('일시', '발전량')
            # 날짜 컬럼 찾기
            date_col = '거래일자' if '거래일자' in df.columns else '일시'
            time_col = '거래시간' if '거래시간' in df.columns else None

            # 필요한 컬럼만 선택
            cols_to_keep = [date_col, solar_col]
            if time_col: cols_to_keep.append(time_col)

            df = df[cols_to_keep].copy()
            df.rename(columns={date_col: '거래일자', solar_col: '발전량', time_col: '거래시간'}, inplace=True)

            # 4. 날짜 형식 변환
            df['거래일자'] = pd.to_datetime(df['거래일자'])

            # 5. 연도 필터링 (통합 파일인 경우 2023년만 추출)
            if ftype == 'mixed':
                df = df[df['거래일자'].dt.year == 2023].copy()

            all_dfs.append(df)
            print(f"  -> 로드 성공: {fname} ({len(df)}개)")

        except Exception as e:
            print(f"   에러 발생 ({fname}): {e}")

    # 최종 병합
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)

        # 날짜순 정렬
        if '거래시간' in final_df.columns:
            final_df.sort_values(by=['거래일자', '거래시간'], inplace=True)
        else:
            final_df.sort_values(by=['거래일자'], inplace=True)

        # 저장
        save_name = 'daegu_solar_2023_test.csv'
        final_df.to_csv(save_name, index=False, encoding='utf-8-sig')

        print("\n" + "="*40)
        print(f"✅ 2023년 전체 데이터 완성! (풍력 제거됨)")
        print(f"파일명: {save_name}")
        print(f"총 데이터: {len(final_df)}개")
        print(f"기간: {final_df['거래일자'].min().date()} ~ {final_df['거래일자'].max().date()}")
        print("="*40)
        print(final_df.head())
    else:
        print("병합할 데이터가 없습니다.")

create_full_2023_dataset()

데이터 병합 시작...

  -> 로드 성공: /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/한국전력거래소_지역별 시간별 태양광 발전량_202301_202302.csv (1416개)
  -> 로드 성공: /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/한국전력거래소_지역별 시간별 태양광 발전량_20230301_20230531.csv (2208개)
  -> 로드 성공: /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/지역별 시간별 태양광 및 풍력 발전량_20230601_20230831.csv (2208개)
  -> 로드 성공: /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/지역별 시간대별 태양광 및 풍력 발전량(2309_2311).csv (2184개)
  -> 로드 성공: /content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/한국전력거래소_지역별 시간별 태양광 및 풍력 발전량_20231231.csv (744개)

✅ 2023년 전체 데이터 완성! (풍력 제거됨)
파일명: daegu_solar_2023_test.csv
총 데이터: 8760개
기간: 2023-01-01 ~ 2023-12-31
        거래일자  발전량  거래시간
0 2023-01-01  0.0     1
1 2023-01-01  0.0     2
2 2023-01-01  0.0     3
3 2023-0

In [ ]:
from google.colab import userdata
asos_api_key = userdata.get('ASOS_API')

print('API 등록', asos_api_key)

In [ ]:
import requests

url = 'http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList'
params ={'serviceKey' : asos_api_key, 'pageNo' : '1', 'numOfRows' : '10', 'dataType' : 'XML', 'dataCd' : 'ASOS', 'dateCd' : 'HR', 'startDt' : '20100101', 'startHh' : '01', 'endDt' : '20100601', 'endHh' : '01', 'stnIds' : '108' }

response = requests.get(url, params=params)
print(response.content)

b'<?xml version="1.0" encoding="UTF-8"?>\r\n<response><header><resultCode>00</resultCode><resultMsg>NORMAL_SERVICE</resultMsg></header><body><dataType>XML</dataType><items><item><tm>2010-01-01 01:00</tm><rnum>1</rnum><stnId>108</stnId><stnNm>\xec\x84\x9c\xec\x9a\xb8</stnNm><ta>-11.0</ta><taQcflg>0</taQcflg><rn></rn><rnQcflg></rnQcflg><ws>1.2</ws><wsQcflg>0</wsQcflg><wd>290</wd><wdQcflg>0</wdQcflg><hm>45</hm><hmQcflg>0</hmQcflg><pv>1.2</pv><td>-20.6</td><pa>1012.5</pa><paQcflg>0</paQcflg><ps>1023.8</ps><psQcflg>0</psQcflg><ss></ss><ssQcflg>9</ssQcflg><icsr></icsr><dsnw>2.2</dsnw><hr3Fhsc></hr3Fhsc><dc10Tca></dc10Tca><dc10LmcsCa></dc10LmcsCa><clfmAbbrCd></clfmAbbrCd><lcsCh></lcsCh><vs></vs><gndSttCd></gndSttCd><dmstMtphNo></dmstMtphNo><ts>-6.5</ts><tsQcflg>0</tsQcflg><m005Te>-4.7</m005Te><m01Te>-2.1</m01Te><m02Te>-0.6</m02Te><m03Te>0.6</m03Te></item><item><tm>2010-01-01 02:00</tm><rnum>2</rnum><stnId>108</stnId><stnNm>\xec\x84\x9c\xec\x9a\xb8</stnNm><ta>-11.1</ta><taQcflg>0</taQcflg><rn>

In [ ]:
import requests
import pandas as pd
from urllib.parse import unquote
import time

# -----------------------------------------------------------
# [입력 필요] 공공데이터포털에서 발급받은 '일반 인증키(Decoding)'를 따옴표 안에 넣어주세요.
# -----------------------------------------------------------
SERVICE_KEY = asos_api_key

def get_daegu_weather_2023(api_key):
    """
    2023년 대구(143) 기상 데이터를 월별로 수집하여 통합하는 함수
    """
    base_url = "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"

    # 수집된 월별 데이터를 담을 리스트
    monthly_data_list = []

    print("데이터 수집을 시작합니다... (대구: 143, 대상 연도: 2023)")

    # 1월부터 12월까지 반복
    for month in range(1, 13):
        # 시작일과 종료일 계산 (예: 20240101 ~ 20240131)
        start_date = f"2023{month:02d}01"

        # 각 월의 마지막 날짜 계산 (간단한 로직)
        if month in [1, 3, 5, 7, 8, 10, 12]:
            last_day = 31
        elif month == 2:
            last_day = 29 # 2024년은 윤년
        else:
            last_day = 30

        end_date = f"2023{month:02d}{last_day}"

        # API 요청 변수 설정
        params = {
            'serviceKey': unquote(api_key),
            'pageNo': '1',
            'numOfRows': '999', # 한 달 최대 시간(744시간)보다 넉넉하게 설정
            'dataType': 'JSON',
            'dataCd': 'ASOS',
            'dateCd': 'HR',
            'startDt': start_date,
            'startHh': '00',
            'endDt': end_date,
            'endHh': '23',
            'stnIds': '143' # [사실] 대구 지점 코드
        }

        try:
            response = requests.get(base_url, params=params)
            data = response.json()

            # 응답 정상 확인
            if data['response']['header']['resultCode'] == '00':
                items = data['response']['body']['items']['item']
                df = pd.DataFrame(items)
                monthly_data_list.append(df)
                print(f"[성공] {month}월 데이터 {len(df)}개 수집 완료")
            else:
                msg = data['response']['header']['resultMsg']
                print(f"[실패] {month}월 데이터 요청 에러: {msg}")

        except Exception as e:
            print(f"[오류] {month}월 처리 중 예외 발생: {e}")

        # 서버 부하 방지를 위한 짧은 대기
        time.sleep(0.2)

    # 데이터 병합 및 저장
    if monthly_data_list:
        final_df = pd.concat(monthly_data_list, ignore_index=True)

        # 분석에 필요한 핵심 컬럼만 선택 및 이름 변경
        # tm:일시, ta:기온, rn:강수량, ws:풍속, hm:습도, icsr:일사량, dc10Tca:전운량
        cols_map = {
            'tm': '일시',
            'ta': '기온',
            'rn': '강수량',
            'ws': '풍속',
            'hm': '습도',
            'icsr': '일사량',
            'dc10Tca': '전운량'
        }

        # 존재하는 컬럼만 선택
        available_cols = [c for c in cols_map.keys() if c in final_df.columns]
        final_df = final_df[available_cols].rename(columns=cols_map)

        # 결측치(NaN) 처리: 강수량과 일사량의 빈 값은 0으로 채움 (맑은 날/밤)
        final_df['강수량'] = final_df['강수량'].fillna(0)
        final_df['일사량'] = final_df['일사량'].fillna(0)
        final_df['전운량'] = final_df['전운량'].fillna(0) # 구름 데이터가 비어있으면 맑음(0)으로 가정

        # CSV 저장
        filename = "/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_weather_2023.csv"
        final_df.to_csv(filename, index=False, encoding='utf-8-sig')

        print("-" * 50)
        print(f"[완료] 총 {len(final_df)}개의 데이터를 '{filename}' 파일로 저장했습니다.")
        print("데이터 미리보기:")
        print(final_df.head())

    else:
        print("[실패] 수집된 데이터가 없습니다. 인증키를 확인해주세요.")

# 함수 실행
if __name__ == "__main__":
    get_daegu_weather_2023(SERVICE_KEY)

데이터 수집을 시작합니다... (대구: 143, 대상 연도: 2023)
[성공] 1월 데이터 744개 수집 완료
[실패] 2월 데이터 요청 에러: DB_ERROR
[성공] 3월 데이터 744개 수집 완료
[성공] 4월 데이터 720개 수집 완료
[성공] 5월 데이터 744개 수집 완료
[성공] 6월 데이터 720개 수집 완료
[성공] 7월 데이터 744개 수집 완료
[성공] 8월 데이터 744개 수집 완료
[성공] 9월 데이터 720개 수집 완료
[성공] 10월 데이터 744개 수집 완료
[성공] 11월 데이터 720개 수집 완료
[성공] 12월 데이터 744개 수집 완료
--------------------------------------------------
[완료] 총 8088개의 데이터를 '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_weather_2023.csv' 파일로 저장했습니다.
데이터 미리보기:
                 일시    기온 강수량   풍속  습도 일사량 전운량
0  2023-01-01 00:00  -1.8      0.0  83       0
1  2023-01-01 01:00  -2.4      0.3  85       0
2  2023-01-01 02:00  -2.8      0.8  86       0
3  2023-01-01 03:00  -2.9      0.7  88       0
4  2023-01-01 04:00  -3.4      0.8  92       0


# 8-extra. [데이터 수리] 누락된 2023년 2월 데이터 추가 수집

기존 수집 과정에서 `DB_ERROR`로 인해 누락된 **2023년 2월 데이터**만 다시 요청하여, 기존 파일(`daegu_weather_2023.csv`)에 합치는 복구 작업을 수행합니다.

### 📋 수행 절차
1.  **기존 파일 로드**: 11개월 치 데이터가 저장된 CSV 파일을 불러옵니다.
2.  **2월 데이터 재요청**: API를 통해 2023년 2월 1일 ~ 2월 28일 데이터를 다시 받아옵니다.
3.  **병합 및 정렬**: 기존 데이터와 2월 데이터를 합친 뒤, 날짜(`일시`) 순서대로 정렬합니다.
4.  **덮어쓰기**: 완성된 데이터를 다시 같은 파일명으로 저장합니다.

In [ ]:
import pandas as pd
import requests
from urllib.parse import unquote
import time
import os

# -----------------------------------------------------------
# [입력] 인증키를 다시 한번 입력해주세요.
# -----------------------------------------------------------
SERVICE_KEY = asos_api_key
# 기존 파일 경로 (사용자 로그 기반)
EXISTING_FILE_PATH = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_weather_2023.csv'

def repair_february_data(api_key):
    # 1. 기존 파일 로드
    if not os.path.exists(EXISTING_FILE_PATH):
        print(f"[오류] 기존 파일을 찾을 수 없습니다: {EXISTING_FILE_PATH}")
        return

    print("기존 데이터 로드 중...")
    df_existing = pd.read_csv(EXISTING_FILE_PATH)
    print(f"기존 데이터 개수: {len(df_existing)}개 (2월 누락됨)")

    # 2. 2월 데이터만 재요청
    print("\n[재시도] 2023년 2월 데이터 요청 중...")

    base_url = "http://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"
    params = {
        'serviceKey': unquote(api_key),
        'pageNo': '1',
        'numOfRows': '999',
        'dataType': 'JSON',
        'dataCd': 'ASOS',
        'dateCd': 'HR',
        'startDt': '20230201', # 2월 1일
        'endDt': '20230228',   # 2월 28일
        'endHh': '23',
        'stnIds': '143'
    }

    try:
        response = requests.get(base_url, params=params)
        data = response.json()

        if data['response']['header']['resultCode'] == '00':
            items = data['response']['body']['items']['item']
            df_feb = pd.DataFrame(items)
            print(f"-> 2월 데이터 {len(df_feb)}개 수집 성공!")

            # 컬럼 매핑 (기존과 동일하게)
            col_map = {
                'tm': '일시', 'ta': '기온', 'rn': '강수량',
                'ws': '풍속', 'hm': '습도', 'icsr': '일사량', 'dc10Tca': '전운량'
            }
            valid_cols = [c for c in col_map.keys() if c in df_feb.columns]
            df_feb = df_feb[valid_cols].rename(columns=col_map)

            # 숫자형 변환 및 결측치 처리
            numeric_cols = ['기온', '강수량', '풍속', '습도', '일사량', '전운량']
            for col in numeric_cols:
                if col in df_feb.columns:
                    df_feb[col] = pd.to_numeric(df_feb[col], errors='coerce')
            df_feb.fillna(0, inplace=True)

            # 3. 병합 및 정렬
            final_df = pd.concat([df_existing, df_feb], ignore_index=True)
            final_df['일시'] = pd.to_datetime(final_df['일시']) # 날짜 형식 변환
            final_df.sort_values(by='일시', inplace=True) # 날짜순 정렬

            # 4. 저장
            final_df.to_csv(EXISTING_FILE_PATH, index=False, encoding='utf-8-sig')

            print("-" * 40)
            print("[수리 완료] 2월 데이터가 정상적으로 추가되었습니다.")
            print(f"최종 데이터 개수: {len(final_df)}개 (기존 8088 + 2월 672 = 8760 예상)")
            print(f"저장 경로: {EXISTING_FILE_PATH}")

        else:
            print(f"[여전한 에러] 서버 응답 코드: {data['response']['header']['resultCode']}")
            print("잠시 후 다시 시도하거나, 기상자료개방포털에서 수동으로 다운로드해야 합니다.")

    except Exception as e:
        print(f"[시스템 에러] {e}")

# 함수 실행
repair_february_data(SERVICE_KEY)

기존 데이터 로드 중...
기존 데이터 개수: 8088개 (2월 누락됨)

[재시도] 2023년 2월 데이터 요청 중...
[여전한 에러] 서버 응답 코드: 02
잠시 후 다시 시도하거나, 기상자료개방포털에서 수동으로 다운로드해야 합니다.


# 9. [검증 준비] 2023년 데이터 병합 및 전처리 (Test Set 생성)

학습된 모델을 검증하기 위해 **2023년 기상 데이터(X)**와 **태양광 발전량 데이터(Y)**를 하나로 합칩니다.

### 📋 수행 작업
1.  **데이터 로드**: `daegu_weather_2023.csv` (기상)와 `daegu_solar_2023_test.csv` (발전량)를 불러옵니다.
2.  **데이터 병합 (Inner Join)**:
    * 두 데이터의 **시간('일시')**을 기준으로 합칩니다.
    * **[중요]** 기상 데이터가 누락된 **2월 데이터는 자동으로 제외**됩니다. (교집합만 남김)
3.  **전처리 적용 (학습 때와 동일하게)**:
    * 단위 변환: 발전량(MWh) -> **kWh** (`* 1000`)
    * 파생변수 추가: `월(Month)`, `시간(Hour)` 생성
    * 노이즈 제거: 밤 시간대(일사량 0) 발전량 0 처리
4.  **최종 저장**: 모델에 바로 넣을 수 있는 `final_test_2023.csv`를 생성합니다.

In [ ]:
import pandas as pd
import numpy as np
import os

# 파일 경로 설정 (Colab 현재 경로에 있다고 가정)
# 만약 드라이브 경로라면 앞에 '/content/drive/MyDrive/...' 붙여주세요.
WEATHER_FILE = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_weather_2023.csv'
SOLAR_FILE = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_solar_2023_test.csv' # 아까 생성한 파일

def prepare_2023_test_set():
    # 1. 파일 로드
    if not os.path.exists(WEATHER_FILE):
        print(f" 기상 파일 없음: {WEATHER_FILE}")
        return
    if not os.path.exists(SOLAR_FILE):
        print(f" 태양광 파일 없음: {SOLAR_FILE}")
        return

    print("데이터 로드 중...")
    df_weather = pd.read_csv(WEATHER_FILE)
    df_solar = pd.read_csv(SOLAR_FILE)

    # 2. 시간 컬럼 형식 통일
    df_weather['일시'] = pd.to_datetime(df_weather['일시'])

    # 태양광 파일은 '거래일자', '거래시간'으로 되어있을 수 있음
    if '거래일자' in df_solar.columns:
        df_solar['거래일자'] = pd.to_datetime(df_solar['거래일자'])
        # 거래시간 1~24를 timedelta로 변환하여 더함 (1시 -> 01:00)
        # 주의: 학습 때와 로직 동일하게 (예: 거래시간 1은 01:00 기상 데이터와 매칭)
        df_solar['일시'] = df_solar['거래일자'] + pd.to_timedelta(df_solar['거래시간'], unit='h')

    # 불필요 컬럼 제거
    df_solar = df_solar[['일시', '발전량']].copy()

    # 3. 데이터 병합 (Inner Join)
    # 기상 데이터가 없는 2월은 여기서 자동으로 탈락됨
    df_final = pd.merge(df_weather, df_solar, on='일시', how='inner')

    print(f"병합 완료: {len(df_final)}개 데이터 (2월 제외됨)")

    # 4. 학습 데이터와 동일한 전처리 (매우 중요!)

    # (1) 단위 변환 (MWh -> kWh)
    # 학습 때 *1000을 했으므로 여기도 똑같이 해줘야 함
    df_final['발전량'] = df_final['발전량'] * 1000

    # (2) 밤 시간대 노이즈 제거
    mask_night = (df_final['일사량'].fillna(0) <= 0)
    df_final.loc[mask_night, '발전량'] = 0

    # (3) 파생변수 생성
    df_final['월'] = df_final['일시'].dt.month
    df_final['시간'] = df_final['일시'].dt.hour

    # (4) 결측치 처리
    df_final = df_final.fillna(0)

    # 5. 저장
    save_name = 'final_test_2023.csv'
    df_final.to_csv(save_name, index=False, encoding='utf-8-sig')

    print("-" * 30)
    print(f" 검증용 데이터셋 준비 완료: {save_name}")
    print(df_final.head())

    # 통계 확인
    print("\n[데이터 통계]")
    print(df_final[['기온', '일사량', '발전량']].describe())

# 실행
prepare_2023_test_set()

데이터 로드 중...
병합 완료: 8087개 데이터 (2월 제외됨)
------------------------------
 검증용 데이터셋 준비 완료: final_test_2023.csv
                   일시   기온  강수량   풍속    습도  일사량  전운량  발전량  월  시간
0 2023-01-01 01:00:00 -2.4  0.0  0.3  85.0  0.0    0  0.0  1   1
1 2023-01-01 02:00:00 -2.8  0.0  0.8  86.0  0.0    0  0.0  1   2
2 2023-01-01 03:00:00 -2.9  0.0  0.7  88.0  0.0    0  0.0  1   3
3 2023-01-01 04:00:00 -3.4  0.0  0.8  92.0  0.0    0  0.0  1   4
4 2023-01-01 05:00:00 -3.3  0.0  0.4  90.0  0.0    0  0.0  1   5

[데이터 통계]
                기온          일사량           발전량
count  8087.000000  8087.000000   8087.000000
mean     16.072264     0.633853   8371.958399
std       9.949548     0.937375  12005.894386
min     -14.000000     0.000000      0.000000
25%       8.800000     0.000000      0.000000
50%      17.700000     0.020000    394.685000
75%      24.100000     1.080000  14589.869500
max      37.600000     3.710000  44386.727000


In [ ]:
WEATHER_FILE = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_weather_2023.csv'
SOLAR_FILE = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_solar_2023_test.csv' # 아까 생성한 파일

df_weather = pd.read_csv(WEATHER_FILE)
df_solar = pd.read_csv(SOLAR_FILE)

print(df_weather.info())
print(df_solar.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8088 entries, 0 to 8087
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   일시      8088 non-null   object 
 1   기온      8088 non-null   float64
 2   강수량     847 non-null    float64
 3   풍속      8088 non-null   float64
 4   습도      8084 non-null   float64
 5   일사량     4446 non-null   float64
 6   전운량     8088 non-null   int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 442.4+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   거래일자    8760 non-null   object 
 1   발전량     8760 non-null   float64
 2   거래시간    8760 non-null   int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 205.4+ KB
None


In [ ]:
import pandas as pd

# CSV 파일 불러오기 (이미 위에서 불러왔지만 명확화를 위해 다시 포함)
file_path = WEATHER_FILE
final_df = pd.read_csv(file_path, encoding='utf-8-sig')

# '강수량'과 '일사량' 컬럼을 숫자로 변환하고 NaN을 0으로 채움
# errors='coerce'를 사용하여 숫자로 변환할 수 없는 값들은 NaN으로 만듦
final_df['강수량'] = pd.to_numeric(final_df['강수량'], errors='coerce').fillna(0)
final_df['일사량'] = pd.to_numeric(final_df['일사량'], errors='coerce').fillna(0)
final_df['전운량'] = pd.to_numeric(final_df['전운량'], errors='coerce').fillna(0)
final_df['습도'] = pd.to_numeric(final_df['전운량'], errors='coerce').fillna(0)

print("결측치 처리 후 DataFrame 정보:")
final_df.info()

결측치 처리 후 DataFrame 정보:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8088 entries, 0 to 8087
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   일시      8088 non-null   object 
 1   기온      8088 non-null   float64
 2   강수량     8088 non-null   float64
 3   풍속      8088 non-null   float64
 4   습도      8088 non-null   int64  
 5   일사량     8088 non-null   float64
 6   전운량     8088 non-null   int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 442.4+ KB


In [ ]:
import pandas as pd
import numpy as np

# 파일명 설정
# weather_file = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction//final_daegu_weather_2024.csv'
solar_file = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/daegu_solar_2023_test.csv'

def create_final_dataset():
    # 1. 데이터 로드
    print("데이터를 불러오는 중...")
    df_weather = final_df
    df_solar = pd.read_csv(solar_file)

    # 2. 기상 데이터 전처리
    # '일시' 컬럼을 문자열에서 datetime 객체로 변환
    df_weather['일시'] = pd.to_datetime(df_weather['일시'])

    # 3. 발전량 데이터 시간 형식 변환
    # 거래일자(String)를 datetime으로 변환
    df_solar['거래일자'] = pd.to_datetime(df_solar['거래일자'])

    # [중요] 거래일자에 거래시간(Hour)을 더해 '일시' 컬럼 생성
    # timedelta를 사용하면 24시도 다음날 00시로 자동 계산되어 안전합니다.
    # 예: 1시 -> 01:00:00 (기상 데이터 01:00와 매칭)
    df_solar['일시'] = df_solar['거래일자'] + pd.to_timedelta(df_solar['거래시간'], unit='h')

    # 불필요한 컬럼 제거 (거래일자, 거래시간, 지역, 연료원 등)
    df_solar = df_solar[['일시', '발전량']]

    # 4. 데이터 병합 (Inner Join)
    # 기상 데이터와 발전량 데이터를 '일시' 기준으로 합침
    print(f"병합 전 - 기상: {len(df_weather)}, 태양광: {len(df_solar)}")
    df_final = pd.merge(df_weather, df_solar, on='일시', how='inner')
    print(f"병합 후 - 최종 데이터: {len(df_final)}")

    # 5. 데이터 정제 (Data Cleaning)
    # [사실] 일사량이 0인 밤에는 태양광 발전이 불가능함.
    # 하지만 ESS 방전 등으로 인해 발전량이 0보다 크게 잡히는 경우가 있음. 이를 0으로 보정.
    # 일사량이 없거나(0), 전운량이 비정상적인 경우 등을 고려

    # 조건: 일사량이 0이거나 NaN인데 발전량이 0보다 큰 경우 -> 발전량을 0으로 강제
    # (참고: 기상청 일사량은 0 또는 NaN일 때 밤인 경우가 많음)
    mask_night = (df_final['일사량'].fillna(0) <= 0)
    df_final.loc[mask_night, '발전량'] = 0

    # 6. 파생 변수 추가 (Feature Engineering)
    # 머신러닝 모델이 '여름', '낮' 등의 패턴을 이해하기 쉽게 숫자 정보 추출
    df_final['월'] = df_final['일시'].dt.month
    df_final['시간'] = df_final['일시'].dt.hour

    # 컬럼 순서 재배치 (보기 좋게)
    # [일시, 월, 시간, 기상변수들..., 발전량(Target)]
    cols = ['일시', '월', '시간', '기온', '강수량', '풍속', '습도', '일사량', '전운량', '발전량']
    df_final = df_final[cols]

    # 결측치 최종 확인 및 0으로 대체 (머신러닝 에러 방지)
    df_final = df_final.fillna(0)

    # 7. 최종 저장
    save_filename = '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/final_dataset_for_AI_test.csv'
    df_final.to_csv(save_filename, index=False, encoding='utf-8-sig')

    print("-" * 50)
    print(f"[완료] 최종 데이터셋 '{save_filename}' 생성 완료")
    print("\n--- 데이터 미리보기 (상위 5개) ---")
    print(df_final.head())

    print("\n--- 데이터 통계 요약 ---")
    print(df_final.describe())

# 함수 실행
if __name__ == "__main__":
    create_final_dataset()

데이터를 불러오는 중...
병합 전 - 기상: 8088, 태양광: 8760
병합 후 - 최종 데이터: 8087
--------------------------------------------------
[완료] 최종 데이터셋 '/content/drive/MyDrive/Colab Notebooks/solar_energy_prediction/final_dataset_for_AI_test.csv' 생성 완료

--- 데이터 미리보기 (상위 5개) ---
                   일시  월  시간   기온  강수량   풍속  습도  일사량  전운량  발전량
0 2023-01-01 01:00:00  1   1 -2.4  0.0  0.3   0  0.0    0  0.0
1 2023-01-01 02:00:00  1   2 -2.8  0.0  0.8   0  0.0    0  0.0
2 2023-01-01 03:00:00  1   3 -2.9  0.0  0.7   0  0.0    0  0.0
3 2023-01-01 04:00:00  1   4 -3.4  0.0  0.8   0  0.0    0  0.0
4 2023-01-01 05:00:00  1   5 -3.3  0.0  0.4   0  0.0    0  0.0

--- 데이터 통계 요약 ---
                                  일시            월           시간           기온  \
count                           8087  8087.000000  8087.000000  8087.000000   
mean   2023-07-13 22:15:33.943365888     6.902807    11.501422    16.072264   
min              2023-01-01 01:00:00     1.000000     0.000000   -14.000000   
25%              2023-04-23 06:30: